# Notebook 03 — Graph Analytics: PageRank + Community Detection

**Tools:** Neo4j GDS (Graph Data Science Library)  
**Goal:** Identify the most *influential* film professionals using network centrality, then detect communities of professionals who work together frequently.

**Business Interpretation:**
- **PageRank** on the crew-collaboration graph reveals who is genuinely central to Hollywood's success network — not just prolific, but *connected to other high-performing professionals*.
- **Louvain Community Detection** reveals natural clusters of crew members who repeatedly work together, revealing the hidden studios/circles that dominate different eras or genres.

> Pre-requisite: Notebook 02 must have been run (graph loaded).

In [ ]:
from neo4j import GraphDatabase
import pandas as pd
import os

URI      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
PASSWORD = os.getenv("NEO4J_PASSWORD", "capstone2024")

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

def run(cypher, label=None, **params):
    with driver.session() as s:
        result = s.run(cypher, **params)
        df = pd.DataFrame([r.data() for r in result])
    if label:
        print(f"\n=== {label} ===")
        if not df.empty: print(df.to_string(index=False))
    return df

run("RETURN 'Connected' AS status", label="Neo4j Status")

## Step 1 — Build a Bipartite Projection: Person ↔ Movie

GDS needs a named graph. We project a bipartite graph of all crew members connected through shared movies. We then use a monopartite co-collaboration graph for PageRank.

In [ ]:
# Drop existing projection if any
try:
    run("CALL gds.graph.drop('crew_collab', false) YIELD graphName")
    print("Previous projection dropped.")
except: pass

# Project bipartite: (Movie) ← all relationship types → (Person nodes)
proj = run("""
CALL gds.graph.project(
  'crew_collab',
  ['Movie', 'Director', 'Actor', 'Producer', 'Writer', 'DOP', 'Composer'],
  {
    DIRECTED_BY: { orientation: 'UNDIRECTED' },
    ACTED_IN:    { orientation: 'UNDIRECTED' },
    PRODUCED_BY: { orientation: 'UNDIRECTED' },
    WRITTEN_BY:  { orientation: 'UNDIRECTED' },
    SHOT_BY:     { orientation: 'UNDIRECTED' },
    SCORE_BY:    { orientation: 'UNDIRECTED' }
  }
)
YIELD graphName, nodeCount, relationshipCount
""", label="Graph Projection")
print(proj)

## Step 2 — PageRank on the Crew-Movie Network

PageRank measures how central each node is in the network. A person with high PageRank is connected to many *other well-connected professionals* — a true industry hub.

In [ ]:
# Write PageRank scores back to nodes
run("""
CALL gds.pageRank.write('crew_collab', {
    maxIterations: 20,
    dampingFactor: 0.85,
    writeProperty: 'pagerank'
})
YIELD nodePropertiesWritten, ranIterations
""", label="PageRank Written")

In [ ]:
# Top PageRank Actors
run("""
    MATCH (a:Actor)
    WHERE a.pagerank IS NOT NULL
    RETURN a.name AS actor, ROUND(a.pagerank, 4) AS pagerank
    ORDER BY pagerank DESC LIMIT 15
""", label="Top PageRank Actors")

In [ ]:
# Top PageRank Directors
run("""
    MATCH (d:Director)
    WHERE d.pagerank IS NOT NULL
    RETURN d.name AS director, ROUND(d.pagerank, 4) AS pagerank
    ORDER BY pagerank DESC LIMIT 15
""", label="Top PageRank Directors")

In [ ]:
# Top PageRank Producers
run("""
    MATCH (p:Producer)
    WHERE p.pagerank IS NOT NULL
    RETURN p.name AS producer, ROUND(p.pagerank, 4) AS pagerank
    ORDER BY pagerank DESC LIMIT 15
""", label="Top PageRank Producers")

In [ ]:
# Top PageRank Writers, DOPs, Composers
for label_type in ["Writer", "DOP", "Composer"]:
    run(f"""
        MATCH (p:{label_type})
        WHERE p.pagerank IS NOT NULL
        RETURN p.name AS name, ROUND(p.pagerank, 4) AS pagerank
        ORDER BY pagerank DESC LIMIT 10
    """, label=f"Top PageRank {label_type}s")

## Step 3 — Louvain Community Detection

**Business interpretation:** Communities reveal natural "circles" or production ecosystems in Hollywood. A community might correspond to:
- A specific studio's regular crew (e.g., Marvel's go-to team)
- An auteur director's recurring collaborators
- A national cinema cluster (e.g., European arthouse filmmakers)

Decision-makers (e.g., casting agents, studio executives) can use this to identify which community a new hire would fit into, or to find untapped cross-community talent.

In [ ]:
# Run Louvain and write community IDs back to nodes
run("""
CALL gds.louvain.write('crew_collab', {
    writeProperty: 'community_id'
})
YIELD communityCount, modularity
""", label="Louvain Communities")

In [ ]:
# Community size distribution (person nodes only, exclude Movie)
run("""
    MATCH (n)
    WHERE n.community_id IS NOT NULL
      AND NOT n:Movie
    WITH n.community_id AS community, COUNT(n) AS size
    ORDER BY size DESC
    RETURN community, size
    LIMIT 20
""", label="Community Size Distribution (Top 20)")

In [ ]:
# Describe top communities: who's in them?
run("""
    MATCH (n)
    WHERE n.community_id IS NOT NULL AND NOT n:Movie
    WITH n.community_id AS community, COUNT(n) AS size
    ORDER BY size DESC LIMIT 5
    WITH COLLECT(community) AS top_communities
    MATCH (n)
    WHERE n.community_id IN top_communities AND NOT n:Movie
    WITH n.community_id AS community, labels(n)[0] AS role, n.name AS name, n.pagerank AS pr
    ORDER BY community, pr DESC
    RETURN community, role, name, ROUND(pr, 4) AS pagerank
    LIMIT 50
""", label="Members of Top 5 Communities")

## Step 4 — Business Interpretation

### PageRank Insights
PageRank on the crew-movie graph reveals professionals who are not just prolific, but deeply embedded in the network of high-quality productions. A director with a PageRank score of 0.5+ has worked with other highly-connected actors and producers, creating a multiplier effect on industry influence.

### Community Insights
The Louvain algorithm reveals distinct production ecosystems:
- **Large communities (100+ members):** Likely reflect major studio ecosystems (e.g., Disney/Marvel's regular crew pool) or genre-based clusters.
- **Medium communities (20-100 members):** Often represent specific auteur directors and their recurring collaborators — the "Nolan circle" or "Spielberg circle".
- **Small communities (<20 members):** Tight-knit indie or foreign language film crews.

**Decision-maker value:** A studio exec can look at a potential hire's community and immediately understand which circle they're from and whether they'd be a cultural/professional fit.

In [ ]:
# Export enriched feature data for ML notebook
import duckdb
from pathlib import Path

df_actors = run("""
    MATCH (a:Actor)<-[:ACTED_IN]-(m:Movie)
    WITH a,
         COUNT(m)              AS degree,
         SUM(m.is_successful)  AS hits,
         AVG(m.vote_average)   AS avg_rating,
         AVG(m.revenue)        AS avg_revenue,
         a.pagerank            AS pagerank,
         a.community_id        AS community_id
    WHERE degree >= 3
    RETURN a.name AS name, degree, hits, 
           ROUND(avg_rating, 3) AS avg_rating,
           ROUND(avg_revenue, 0) AS avg_revenue,
           ROUND(pagerank, 6) AS pagerank,
           community_id,
           ROUND(100.0 * hits / degree, 1) AS hit_rate_pct
""")

DATA_DIR = Path("../data")
df_actors.to_parquet(DATA_DIR / "actor_features.parquet", index=False)
print(f"Actor feature matrix saved: {len(df_actors):,} rows")
df_actors.head(10)

In [ ]:
driver.close()
print("Done.")